# 🌊 Flood Alert Lakehouse · Itaverá
**Data Protection & Ingestion Solutions** · FIAP Global Solution 2026.2

🇬🇧 End-to-end data pipeline for a flood early-warning system built for the city's Civil Defense. It ingests river-level sensors and citizen reports, cleans and deduplicates the data through a medallion architecture (Bronze → Silver → Gold), pseudonymizes personal data in line with Brazil's LGPD, and recovers from an operational error using Delta Lake time travel.

🇧🇷 Pipeline de dados para um sistema de alerta de enchentes da Defesa Civil de Itaverá. Ingere leituras de sensores de rio e denúncias de cidadãos, limpa e deduplica os dados numa arquitetura medalhão (Bronze → Prata → Ouro), pseudonimiza dados pessoais conforme a LGPD e recupera um erro operacional com o time travel do Delta Lake.

**Stack:** Databricks · Spark SQL · Delta Lake · Unity Catalog

**Team / Equipe:** Anna Luiza Torra · João Henrique Arantes · Thalita Soares

---

---
# Q1 · Ambiente (0,5)

In [0]:
   %sql
   CREATE SCHEMA IF NOT EXISTS workspace.defesa_civil;
   CREATE VOLUME IF NOT EXISTS workspace.defesa_civil.gs;

In [0]:

base = "/Volumes/workspace/defesa_civil/gs"
for p in ["estoque/sensores","estoque/denuncias","estoque/cadastros",
          "chegada/sensores","chegada/denuncias","_controle"]:
    dbutils.fs.mkdirs(f"{base}/{p}")
print("pastas criadas")

In [0]:
for p in ["sensores","denuncias","cadastros"]:
    print(f"--- estoque/{p}")
    display(dbutils.fs.ls(f"{base}/estoque/{p}"))

📝 **RESPOSTA Q1:** evidência dos arquivos no lugar.
- Os 6 arquivos estão em estoque/ (saída acima); chegada/ está vazia e _controle foi criada.


---
# Q2 · Bronze dos sensores, incremental (1,5)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.defesa_civil.bronze_sensores;

In [0]:
base = "/Volumes/workspace/defesa_civil/gs"
dbutils.fs.cp(f"{base}/estoque/sensores/sensores_lote1.csv", f"{base}/chegada/sensores/sensores_lote1.csv")
display(dbutils.fs.ls(f"{base}/chegada/sensores"))

In [0]:
%sql
COPY INTO workspace.defesa_civil.bronze_sensores
FROM (
  SELECT *, _metadata.file_path AS arquivo_origem, current_timestamp() AS carregado_em
  FROM '/Volumes/workspace/defesa_civil/gs/chegada/sensores'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true');

In [0]:
dbutils.fs.cp(f"{base}/estoque/sensores/sensores_lote2.csv", f"{base}/chegada/sensores/sensores_lote2.csv")
display(dbutils.fs.ls(f"{base}/chegada/sensores"))

In [0]:
   %sql
   COPY INTO workspace.defesa_civil.bronze_sensores
   FROM (
     SELECT *, _metadata.file_path AS arquivo_origem, current_timestamp() AS carregado_em
     FROM '/Volumes/workspace/defesa_civil/gs/chegada/sensores'
   )
   FILEFORMAT = CSV
   FORMAT_OPTIONS ('header' = 'true')
   COPY_OPTIONS ('mergeSchema' = 'true');

In [0]:
%sql
COPY INTO workspace.defesa_civil.bronze_sensores
FROM (
  SELECT *, _metadata.file_path AS arquivo_origem, current_timestamp() AS carregado_em
  FROM '/Volumes/workspace/defesa_civil/gs/chegada/sensores'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true');

In [0]:
%sql
SELECT element_at(split(arquivo_origem, '/'), -1) AS arquivo,
       count(*) AS linhas_total,
       count_if(data_hora LIKE '2026-02-28%') AS linhas_28_02
FROM workspace.defesa_civil.bronze_sensores
GROUP BY 1 ORDER BY 1;

📝 **RESPOSTA Q2:** linhas de cada uma das três execuções e o porquê da terceira.
- Na execução 1 (lote 1) entraram 32.256 linhas, na execução 2 (lote 2) entraram 36.864 linhas e na execução 3 entraram 0 linhas. A terceira não inseriu nada porque o COPY INTO registra quais arquivos já foram carregados e ignora os que já processou. Por isso a ingestão pode ser reexecutada sem duplicar dados (é idempotente).

Sobre o dia 28/02: a Bronze guardou as leituras desse dia duas vezes, 1.152 vindas do lote 1 e 1.152 do lote 2 (conferência na última célula). Isso está certo: a Bronze é a cópia fiel de tudo o que chegou, só recebe acréscimos e controla por arquivo, não por linha. O lote 2 é um reenvio do fornecedor para o dia 28/02, e essas duplicatas são removidas na camada Prata com MERGE, que mantém uma linha por estação e data/hora (Q4).


---
# Q3 · Bronze das denúncias (1,0)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.defesa_civil.bronze_denuncias;

In [0]:
base = "/Volumes/workspace/defesa_civil/gs"
dbutils.fs.cp(f"{base}/estoque/denuncias/denuncias_lote1.csv", f"{base}/chegada/denuncias/denuncias_lote1.csv")

In [0]:
%sql
COPY INTO workspace.defesa_civil.bronze_denuncias
FROM (
  SELECT *, _metadata.file_path AS arquivo_origem, current_timestamp() AS carregado_em
  FROM '/Volumes/workspace/defesa_civil/gs/chegada/denuncias'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true');

In [0]:
   base = "/Volumes/workspace/defesa_civil/gs"
   dbutils.fs.cp(f"{base}/estoque/denuncias/denuncias_lote2.csv", f"{base}/chegada/denuncias/denuncias_lote2.csv")
   display(dbutils.fs.ls(f"{base}/chegada/denuncias"))

In [0]:
%sql
SELECT element_at(split(arquivo_origem, '/'), -1) AS arquivo, count(*) AS linhas
FROM workspace.defesa_civil.bronze_denuncias
GROUP BY 1;

In [0]:
%sql
SELECT count(*) AS linhas, count(DISTINCT protocolo) AS protocolos_distintos,
       count(*) - count(DISTINCT protocolo) AS repetidos
FROM workspace.defesa_civil.bronze_denuncias;

In [0]:
%sql
COPY INTO workspace.defesa_civil.bronze_denuncias
FROM (
  SELECT *, _metadata.file_path AS arquivo_origem, current_timestamp() AS carregado_em
  FROM '/Volumes/workspace/defesa_civil/gs/chegada/denuncias'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true');

In [0]:
%sql
SELECT element_at(split(arquivo_origem, '/'), -1) AS arquivo, count(*) AS linhas
FROM workspace.defesa_civil.bronze_denuncias GROUP BY 1;

📝 **RESPOSTA Q3:** total de linhas, protocolos distintos e por que diferem; colunas com dados pessoais e quem deve acessar a Bronze.
- A Bronze tem 927 linhas (219 do lote 1 + 708 do lote 2) e 875 protocolos distintos. A diferença de 52 existe porque cada exportação traz o estado atual de todas as denúncias, então os protocolos já presentes em 01/03 voltaram em 01/04 com o status atualizado.
Dados pessoais: nome, cpf, telefone e email, além de latitude/longitude, que podem indicar o endereço. Só a equipe de atendimento, que precisa recontatar o cidadão, e a engenharia que opera o pipeline deveriam acessar a Bronze; analistas e o painel público usam a Prata, que não tem esses dados.
A célula do COPY INTO do lote 2 mostra 0 porque foi reexecutada depois da carga; a conferência por arquivo comprova que as 708 linhas do lote 2 entraram.


---
# Q4 · Qualidade e Prata dos sensores (2,0)

In [0]:
%sql
SELECT upper(trim(estacao)) AS estacao,
       count_if(try_cast(nivel_m AS DOUBLE) < 0) AS nivel_negativo,
       array_distinct(collect_list(CASE WHEN try_cast(nivel_m AS DOUBLE) < 0 THEN nivel_m END)) AS valor_negativo,
       count_if(nivel_m IS NULL OR trim(nivel_m) = '') AS nivel_vazio,
       count_if(estacao <> upper(trim(estacao))) AS codigo_fora_padrao
FROM workspace.defesa_civil.bronze_sensores
GROUP BY 1 ORDER BY 1;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.defesa_civil.prata_estacoes AS
SELECT upper(trim(codigo)) AS estacao, curso_dagua, upper(trim(bairro)) AS bairro,
       CAST(cota_alerta_m AS DOUBLE) AS cota_alerta, CAST(cota_transbordo_m AS DOUBLE) AS cota_transbordo
FROM read_files('/Volumes/workspace/defesa_civil/gs/estoque/cadastros/estacoes.csv', format => 'csv', header => true);

CREATE OR REPLACE TABLE workspace.defesa_civil.prata_bairros AS
SELECT upper(trim(bairro)) AS bairro, CAST(populacao AS INT) AS populacao, area_de_risco,
       CAST(domicilios_com_internet_pct AS INT) AS pct_internet
FROM read_files('/Volumes/workspace/defesa_civil/gs/estoque/cadastros/bairros.csv', format => 'csv', header => true);

SELECT * FROM workspace.defesa_civil.prata_estacoes;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.defesa_civil.prata_sensores
  (estacao STRING, data_hora TIMESTAMP, nivel_m DOUBLE, chuva_mm DOUBLE);

In [0]:
%sql
MERGE INTO workspace.defesa_civil.prata_sensores t
USING (
  SELECT estacao, data_hora, nivel_m, chuva_mm FROM (
    SELECT upper(trim(estacao)) AS estacao, to_timestamp(data_hora) AS data_hora,
           try_cast(nivel_m AS DOUBLE) AS nivel_m, try_cast(chuva_mm AS DOUBLE) AS chuva_mm,
           row_number() OVER (PARTITION BY upper(trim(estacao)), to_timestamp(data_hora)
                              ORDER BY carregado_em DESC) AS rn
    FROM workspace.defesa_civil.bronze_sensores
    WHERE try_cast(nivel_m AS DOUBLE) BETWEEN 0 AND 15
      AND try_cast(chuva_mm AS DOUBLE) BETWEEN 0 AND 100
  ) WHERE rn = 1
) s
ON t.estacao = s.estacao AND t.data_hora = s.data_hora
WHEN NOT MATCHED THEN INSERT *;

In [0]:
%sql
MERGE INTO workspace.defesa_civil.prata_sensores t
USING (
  SELECT estacao, data_hora, nivel_m, chuva_mm FROM (
    SELECT upper(trim(estacao)) AS estacao, to_timestamp(data_hora) AS data_hora,
           try_cast(nivel_m AS DOUBLE) AS nivel_m, try_cast(chuva_mm AS DOUBLE) AS chuva_mm,
           row_number() OVER (PARTITION BY upper(trim(estacao)), to_timestamp(data_hora)
                              ORDER BY carregado_em DESC) AS rn
    FROM workspace.defesa_civil.bronze_sensores
    WHERE try_cast(nivel_m AS DOUBLE) BETWEEN 0 AND 15
      AND try_cast(chuva_mm AS DOUBLE) BETWEEN 0 AND 100
  ) WHERE rn = 1
) s
ON t.estacao = s.estacao AND t.data_hora = s.data_hora
WHEN NOT MATCHED THEN INSERT *;

In [0]:
%sql
SELECT count(*) AS validas_na_bronze,
       (SELECT count(*) FROM workspace.defesa_civil.prata_sensores) AS linhas_prata,
       count(*) - (SELECT count(*) FROM workspace.defesa_civil.prata_sensores) AS duplicatas_removidas
FROM workspace.defesa_civil.bronze_sensores
WHERE try_cast(nivel_m AS DOUBLE) BETWEEN 0 AND 15 AND try_cast(chuva_mm AS DOUBLE) BETWEEN 0 AND 100;

In [0]:
%sql
WITH d AS (
  SELECT upper(trim(estacao)) est, data_hora,
         array_join(array_sort(collect_set(element_at(split(arquivo_origem,'/'),-1))), ' + ') AS arquivos
  FROM workspace.defesa_civil.bronze_sensores
  WHERE try_cast(nivel_m AS DOUBLE) BETWEEN 0 AND 15 AND try_cast(chuva_mm AS DOUBLE) BETWEEN 0 AND 100
  GROUP BY 1,2 HAVING count(*) > 1
)
SELECT arquivos, count(*) AS chaves_duplicadas, min(data_hora) AS de, max(data_hora) AS ate
FROM d GROUP BY 1;

📝 **RESPOSTA Q4:** problemas encontrados por estação; linhas da Prata; duplicatas removidas e de onde vieram.
- Problemas por estação: CP-02 com 4.608 códigos fora do padrão (" cp-02" em todo o lote 2); RP-04 com 45 leituras de nível negativo, todas -999 (falha do transmissor); RP-01 com 42 leituras de nível vazio (manutenção programada). A Prata ficou com 67.881 linhas, uma por estação e data/hora; a segunda execução do MERGE inseriu 0. Foram removidas 1.152 duplicatas, todas do dia 28/02, que veio nos dois lotes (8 estações × 144 leituras): é o reenvio do fornecedor.

---
# Q5 · Prata das denúncias, sem dados pessoais (1,5)

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.defesa_civil.prata_denuncias
  (protocolo STRING, criado_em TIMESTAMP, atualizado_em TIMESTAMP, status STRING, tipo STRING,
   bairro STRING, descricao STRING, latitude DOUBLE, longitude DOUBLE, cidadao_hash STRING);

In [0]:
%sql
MERGE INTO workspace.defesa_civil.prata_denuncias t
USING (
  SELECT * EXCEPT (rn) FROM (
    SELECT protocolo, to_timestamp(criado_em) criado_em, to_timestamp(atualizado_em) atualizado_em,
           status, tipo, upper(trim(bairro)) bairro, descricao,
           CAST(latitude AS DOUBLE) latitude, CAST(longitude AS DOUBLE) longitude,
           sha2(cpf, 256) AS cidadao_hash,
           row_number() OVER (PARTITION BY protocolo ORDER BY carregado_em DESC, atualizado_em DESC) rn
    FROM workspace.defesa_civil.bronze_denuncias
  ) WHERE rn = 1
) s
ON t.protocolo = s.protocolo
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
%sql
SELECT count(*) AS linhas, count(DISTINCT cidadao_hash) AS cidadaos_distintos
FROM workspace.defesa_civil.prata_denuncias;

In [0]:
%sql
SELECT cidadao_hash, bairro, count(*) AS denuncias
FROM workspace.defesa_civil.prata_denuncias
GROUP BY 1,2 ORDER BY denuncias DESC LIMIT 5;

📝 **RESPOSTA Q5:** linhas da Prata; cidadãos distintos; maior número de denúncias de um mesmo cidadão e o bairro; por que usar o hash em vez de apagar o CPF.
- A Prata tem 875 linhas, uma por protocolo, com a versão mais recente. 335 cidadãos distintos fizeram denúncias, e o que mais denunciou fez 54, no Jardim Várzea. Nome, e-mail e telefone não foram levados para a Prata.
O hash é melhor do que apagar o CPF porque o mesmo CPF sempre gera o mesmo código: dá para contar cidadãos distintos e identificar denúncias repetidas sem saber quem é a pessoa. Apagando, essa análise se perde. Ressalva: hash de CPF sem salt é pseudonimização, e como existem poucos CPFs possíveis ele pode ser revertido por força bruta; pela LGPD continua sendo dado pessoal e exige controle de acesso.

---
# Q6 · Análises e Ouro (2,0)

In [0]:
%sql
SELECT estacao, to_date(data_hora) AS dia, round(sum(chuva_mm),1) AS chuva_dia
FROM workspace.defesa_civil.prata_sensores
GROUP BY 1,2 HAVING sum(chuva_mm) > 50 ORDER BY dia, estacao;

In [0]:
%sql
SELECT count(DISTINCT dia) AS temporais, array_sort(collect_set(dia)) AS datas FROM (
  SELECT estacao, to_date(data_hora) dia FROM workspace.defesa_civil.prata_sensores
  GROUP BY 1,2 HAVING sum(chuva_mm) > 50);

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.defesa_civil.ouro_horas_acima_cota AS
SELECT e.estacao, e.bairro,
       round(count_if(s.nivel_m >= e.cota_alerta)/6.0, 2)     AS horas_alerta,
       round(count_if(s.nivel_m >= e.cota_transbordo)/6.0, 2) AS horas_transbordo
FROM workspace.defesa_civil.prata_estacoes e
LEFT JOIN workspace.defesa_civil.prata_sensores s ON s.estacao = e.estacao
GROUP BY 1,2;
SELECT * FROM workspace.defesa_civil.ouro_horas_acima_cota ORDER BY horas_transbordo DESC;

In [0]:
%sql
SELECT e.estacao, e.bairro, count(s.data_hora) AS leituras_validas_09_03, max(s.nivel_m) AS nivel_max,
       round(count_if(s.nivel_m >= e.cota_transbordo)/6.0, 2) AS horas_transbordo_09_03
FROM workspace.defesa_civil.prata_estacoes e
LEFT JOIN workspace.defesa_civil.prata_sensores s
  ON s.estacao = e.estacao AND to_date(s.data_hora) = '2026-03-09'
GROUP BY 1,2 ORDER BY 1;

In [0]:
%sql
WITH alerta AS (
  SELECT e.bairro, e.estacao, min(s.data_hora) AS primeiro_alerta
  FROM workspace.defesa_civil.prata_sensores s JOIN workspace.defesa_civil.prata_estacoes e ON s.estacao = e.estacao
  WHERE s.data_hora >= '2026-03-09 12:00' AND s.data_hora < '2026-03-10 12:00' AND s.nivel_m >= e.cota_alerta
  GROUP BY 1,2),
den AS (
  SELECT bairro, min(criado_em) AS primeira_denuncia FROM workspace.defesa_civil.prata_denuncias
  WHERE tipo = 'alagamento' AND criado_em >= '2026-03-09 12:00' AND criado_em < '2026-03-10 12:00'
  GROUP BY 1)
SELECT coalesce(a.bairro, d.bairro) AS bairro, a.estacao, a.primeiro_alerta, d.primeira_denuncia,
       a.primeiro_alerta < d.primeira_denuncia AS alerta_veio_antes
FROM alerta a FULL OUTER JOIN den d ON a.bairro = d.bairro ORDER BY primeira_denuncia;

In [0]:
%sql
SELECT estacao, data_hora, nivel_m, chuva_mm FROM workspace.defesa_civil.bronze_sensores
WHERE upper(trim(estacao)) = 'RP-04' AND data_hora BETWEEN '2026-03-09 18:00' AND '2026-03-10 00:30'
ORDER BY data_hora;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.defesa_civil.ouro_denuncias_bairro AS
SELECT b.bairro, b.populacao, b.pct_internet,
       count(d.protocolo) AS denuncias,
       round(count(d.protocolo) * 10000.0 / b.populacao, 1) AS denuncias_10mil,
       count(DISTINCT d.cidadao_hash) AS cidadaos,
       round(count(DISTINCT d.cidadao_hash) * 10000.0 / b.populacao, 1) AS cidadaos_10mil,
       b.bairro IN (SELECT bairro FROM workspace.defesa_civil.prata_estacoes) AS tem_estacao
FROM workspace.defesa_civil.prata_bairros b
LEFT JOIN workspace.defesa_civil.prata_denuncias d
  ON d.bairro = b.bairro AND d.tipo IN ('alagamento', 'deslizamento')
GROUP BY b.bairro, b.populacao, b.pct_internet;
SELECT * FROM workspace.defesa_civil.ouro_denuncias_bairro ORDER BY denuncias_10mil DESC;

In [0]:
%sql
SELECT o.bairro, h.estacao, h.horas_transbordo, o.denuncias_10mil, o.pct_internet
FROM workspace.defesa_civil.ouro_denuncias_bairro o
JOIN workspace.defesa_civil.ouro_horas_acima_cota h ON h.bairro = o.bairro
WHERE h.horas_transbordo > 0 ORDER BY o.denuncias_10mil;

📝 **RESPOSTA Q6:**
- a) Houve 3 temporais: 14/02, 09/03 e 25/03. O de 09/03 foi o maior, com mais de 100 mm no dia em quase todas as estações.

- b) A estação que ficou mais tempo acima do transbordo foi a RP-03 (Centro), com 5,5 h. A RP-04 (Beira-Rio) aparece com 0 h de transbordo em 09/03, mesmo no maior temporal. A questão 4 explica o motivo: as leituras -999 da falha do transmissor foram descartadas e ela tem só 111 leituras válidas no dia, contra 144 das outras. RP-01 e RP-02 também têm 0, mas com dados completos: o nível máximo ficou abaixo da cota de transbordo.

- c) As estações deram o primeiro alerta entre 18h50 e 20h00, e as primeiras denúncias de alagamento chegaram entre 19h44 e 20h53. Nos bairros com estação funcionando, o alerta veio antes. No Beira-Rio, a denúncia chegou às 20h32 e o “alerta” só aparece às 00h00 de 10/03, quando o transmissor voltou: entre 18h30 e 23h50 a RP-04 enviou -999. O zero de transbordo significa “sem dado”, não “sem cheia”. Quando uma estação para de transmitir durante um temporal, a Defesa Civil deveria tratar a falta de dados como alerta: monitorar a atualização das leituras, acionar equipe em campo, usar estações vizinhas e as denúncias como sinal complementar, e ter telemetria redundante. Dado ausente nunca pode ser lido como nível normal.

- d) O líder é o Jardim Várzea, com 557,7 denúncias por 10 mil habitantes, e ele não tem estação. A recomendação é instalar um sensor lá. Contando cidadãos distintos, o Jardim Várzea continua liderando (150 por 10 mil), mas a taxa cai muito: um único cidadão fez 54 denúncias, o que mostra que contar denúncias superestima o problema.

- e) Entre os bairros com transbordo, a Vila Esperança (CP-03) tem a menor taxa, 10,5 por 10 mil, e o menor percentual de domicílios com internet, 38%. Poucas denúncias ali indicam exclusão digital, não ausência de problema. Uma solução que dependa só do app deixa de enxergar justamente as populações mais vulneráveis; por isso os sensores e canais como telefone, SMS e agentes comunitários são necessários.

---
# Q7 · Erro de operação e recuperação (1,5)

In [0]:
%sql
SELECT status, count(*) AS qtd FROM workspace.defesa_civil.prata_denuncias GROUP BY 1 ORDER BY 1;

In [0]:
%sql
UPDATE workspace.defesa_civil.prata_denuncias
SET status = 'concluída'
WHERE status IN ('aberta', 'em atendimento');

In [0]:
%sql
DESCRIBE HISTORY workspace.defesa_civil.prata_denuncias;

In [0]:
   %sql
   SELECT 'v4 - depois do erro' AS versao, status, count(*) qtd FROM workspace.defesa_civil.prata_denuncias VERSION AS OF 4 GROUP BY 1,2
   UNION ALL
   SELECT 'v3 - antes do erro', status, count(*) FROM workspace.defesa_civil.prata_denuncias VERSION AS OF 3 GROUP BY 1,2
   ORDER BY 1,2;

In [0]:
%sql
   RESTORE TABLE workspace.defesa_civil.prata_denuncias TO VERSION AS OF 3;

In [0]:
   %sql
   SELECT status, count(*) AS qtd FROM workspace.defesa_civil.prata_denuncias GROUP BY 1 ORDER BY 1;

📝 **RESPOSTA Q7:** linhas alteradas pelo erro; versão restaurada; validação; quando o histórico não basta.
- Antes do erro: concluída 828, em atendimento 44, aberta 3. O UPDATE alterou 47 linhas. Pelo DESCRIBE HISTORY, a versão anterior ao erro é a 3, e a tabela foi restaurada para ela. A validação mostra que a contagem voltou a 828 / 44 / 3.
O RESTORE funcionou sem backup porque o Delta não sobrescreve arquivos: o UPDATE gravou arquivos novos e só marcou os antigos como removidos no log de transações (_delta_log). Os arquivos antigos continuaram no armazenamento, e o RESTORE fez o log voltar a apontar para eles.
O histórico deixa de funcionar quando:

um VACUUM apaga os arquivos antigos depois do prazo de retenção (padrão de 7 dias);
o log expira (30 dias);
a tabela ou o armazenamento é apagado, corrompido ou atingido por ransomware.

Como o histórico fica no mesmo lugar que os dados, time travel não substitui um backup isolado (regra 3-2-1).